# Data Notebook — IP & PMI Pricing Artifacts
### ABC Health | 9th IAI Capacity Building Seminar in Health and Care Insurance

This notebook does the data generation for **both** products used in notebook 05 (`05_pricing_team.ipynb`), so that notebook can focus purely on tools and agents with **no data generation of its own** — exactly the same separation of concerns as `build_data.py` in the companion Streamlit app.

**Run this once.** It produces two files:
- `ip_pricing_artifacts.pkl` — population, spells, and Tool 1/2/3 tables, generated with the **exact same code as notebook 04** (copied verbatim, not re-derived), so results are identical to notebook 04's.
- `pmi_pricing_artifacts.pkl` — a synthetic PMI policy/claims dataset plus frequency, severity, and NCB tables, calibrated to the canonical portfolio figures used across this seminar's materials (6% frequency, ₹85,000 severity, ₹5,100 pure premium).

Before running notebook 05: place both `.pkl` files in the same working directory (or the shared repo's `data/` folder, if running across separate Colab sessions).

## 0. Setup

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)

OUTPUT_DIR = Path('.')

## Part A — IP Artifacts

Every constant and function below is copied **verbatim** from notebook 04 — same seeds, same logic, same results. If you've already run notebook 04, this section reproduces exactly what you saw there; nothing here is a re-derivation.

In [2]:
DEFERRED_WEEKS = 13
STANDARD_DEFERRED_OPTIONS = [4, 13, 26, 52]   # the deferred-period choices actually priced
STUDY_WEEKS = 520          # 10-year observation window (see note above on why not 5 years)
N_LIVES = 6000

# Age-banded base incidence — steadily increasing with age, this is a rating factor that
# lives in Tool 1's pooled base table (25-34 / 35-49 / 50-60).
AGE_BAND_LABELS = {0: '25-34', 1: '35-49', 2: '50-60'}
AGE_BASE_INCIDENCE = {0: 0.035, 1: 0.055, 2: 0.085}


def age_band(age):
    """Bands age into the three rating groups. Anyone below 35 -> band 0, 35-49 -> band 1,
    50 and above -> band 2 (naturally clamps any age, no invented band possible)."""
    if age < 35:
        return 0
    elif age < 50:
        return 1
    return 2


# Occupation class — a couple of high-level categories, the second rating factor in Tool 1's
# base table (same treatment as age: it affects INCIDENCE only, not deferred-recovery or
# claiming-duration behaviour, which stay governed purely by episode history in Tool 3).
OCCUPATION_CLASSES = ['desk', 'manual']
OCCUPATION_MULT = {'desk': 1.00, 'manual': 1.55}
OCCUPATION_MIX = {'desk': 0.65, 'manual': 0.35}   # proportion of the simulated population in each class

# Episode-band incidence MULTIPLIER — applied on top of the age/occupation base rate above,
# capped at band "2+" (originally split into 0/1/2/3+, but band 3 turned out too thin — 9 spells
# — for the credibility weighting to produce a stable loading. Capping one band earlier pools
# band 2 and band 3+ together, giving a much more solidly estimated top band.) Multipliers
# preserve the same relative escalation as the original flat rates (2.4x, 3.3x).
EPISODE_INCIDENCE_MULT = {0: 1.00, 1: 2.40, 2: 3.30}

# TOTAL sickness-spell duration (deferred + claiming combined) is now drawn as ONE underlying
# duration per spell, from onset to eventual recovery/death — the deferred period is then applied
# as a genuine CUTOFF on that single duration, not a separate independent draw. This is what
# makes deferred_weeks an actual lever: a shorter deferred period pulls more of the same
# underlying spells into paid claiming, a longer one keeps more of them fully within the
# (unpaid) deferred window. Same declining-hazard Weibull shape as before, scaled up by episode
# history (repeat episodes take longer to resolve overall, not just once claiming starts).
TERMINAL_DEATH_PROB = 0.020        # probability the whole spell ends in death, not recovery
TOTAL_DURATION_BASE_SCALE_WEEKS = 10.0
EPISODE_DURATION_SCALE_MULT = {0: 1.00, 1: 1.30, 2: 1.50}
CLAIMING_WEIBULL_SHAPE = 0.60      # shape < 1 -> declining hazard, i.e. "stickier" the longer you're sick
MAX_SPELL_WEEKS = 156              # overall cap on total sickness duration (3 years)


def episode_band(prior_episode_count):
    """Caps episode history at band 2, matching the credible-band logic agreed earlier —
    we never model a smooth curve into thin data past episode 2."""
    return min(prior_episode_count, 2)


def generate_population(n=N_LIVES, seed=42):
    rng = np.random.default_rng(seed)
    ages = rng.integers(25, 61, size=n)
    incomes = rng.lognormal(mean=np.log(60000), sigma=0.45, size=n).round(-2)
    incomes = np.clip(incomes, 20000, 400000)
    occupations = rng.choice(OCCUPATION_CLASSES, size=n,
                              p=[OCCUPATION_MIX[c] for c in OCCUPATION_CLASSES])
    return pd.DataFrame({
        'policyholder_id': [f'PH{str(i).zfill(5)}' for i in range(n)],
        'age': ages,
        'monthly_income': incomes.astype(int),
        'occupation': occupations,
    })


ip_population = generate_population()
print(f"IP population: {len(ip_population):,} lives")
ip_population.head()

IP population: 6,000 lives


,policyholder_id,age,monthly_income,occupation
0,PH00000,28,96500,desk
1,PH00001,52,42700,manual
2,PH00002,48,20000,desk
3,PH00003,40,27100,desk
4,PH00004,40,113800,desk


In [3]:
def simulate_spells(population, study_weeks=STUDY_WEEKS, deferred_weeks=DEFERRED_WEEKS, seed=7):
    """Simulates sickness spells per life: onset -> Sick(deferred) -> [Sick(claiming)] -> resolution.
    This is the raw claims-generating process; the pricing tools only ever see the output of
    this function, exactly as a real pricing team would only see claims data, not the
    underlying (unobservable) hazard process.

    Each spell draws ONE underlying total-sickness-duration (onset to eventual recovery/death) —
    the deferred period is then applied as a genuine cutoff on that single duration, not a
    separate independent draw. This is what makes deferred_weeks a real lever, and it's also
    exactly why Tool 2 doesn't need to call this function again for other deferred-period
    options: since the full duration is recorded for every spell regardless of outcome, any
    other deferred-period cutoff can be applied directly to this one dataset afterward — see
    Tool 2 in notebook 04."""
    rng = np.random.default_rng(seed)
    records = []

    for _, ph in population.iterrows():
        week = 0.0
        episode_number = 0
        alive = True
        a_band = age_band(ph['age'])
        occ = ph['occupation']

        while alive and week < study_weeks:
            band = episode_band(episode_number)
            p_annual = AGE_BASE_INCIDENCE[a_band] * OCCUPATION_MULT[occ] * EPISODE_INCIDENCE_MULT[band]
            weekly_hazard = -np.log(1 - p_annual) / 52
            wait = rng.exponential(1 / weekly_hazard)
            week_onset = week + wait
            if week_onset >= study_weeks:
                break

            episode_number += 1

            is_death = rng.random() < TERMINAL_DEATH_PROB
            scale = TOTAL_DURATION_BASE_SCALE_WEEKS * EPISODE_DURATION_SCALE_MULT[band]
            resolution_week = float(rng.weibull(CLAIMING_WEIBULL_SHAPE) * scale)
            resolution_week = min(resolution_week, MAX_SPELL_WEEKS)

            if resolution_week <= deferred_weeks:
                deferred_outcome = 'died_in_deferred' if is_death else 'recovered_in_deferred'
                weeks_in_deferred = resolution_week
                claiming_weeks = 0.0
                claim_outcome = 'na'
            else:
                deferred_outcome = 'crossed_to_claiming'
                weeks_in_deferred = float(deferred_weeks)
                claiming_weeks = resolution_week - deferred_weeks
                claim_outcome = 'died_while_claiming' if is_death else 'recovered_while_claiming'
            if is_death:
                alive = False

            week_end = week_onset + weeks_in_deferred + claiming_weeks
            censored = week_end > study_weeks
            if censored:
                overrun = week_end - study_weeks
                claiming_weeks = max(0.0, claiming_weeks - overrun)
                week_end = study_weeks

            monthly_claim_amount = round(0.8 * ph['monthly_income'], 2)
            total_claim_paid = round((claiming_weeks / 4.345) * monthly_claim_amount, 2)

            records.append({
                'policyholder_id': ph['policyholder_id'],
                'age_at_onset': ph['age'],
                'age_band': a_band,
                'occupation': occ,
                'monthly_income': ph['monthly_income'],
                'episode_number': episode_number,
                'episode_band': band,
                'week_onset': round(week_onset, 2),
                'deferred_outcome': deferred_outcome,
                'weeks_in_deferred': round(weeks_in_deferred, 2),
                'crossed_to_claiming': deferred_outcome == 'crossed_to_claiming',
                'claiming_weeks': round(claiming_weeks, 2),
                'claim_outcome': claim_outcome,
                'monthly_claim_amount': monthly_claim_amount,
                'total_claim_paid': total_claim_paid,
                'censored': censored,
            })
            week = week_end

    return pd.DataFrame(records)


ip_spells = simulate_spells(ip_population)
print(f"Total sickness spells simulated : {len(ip_spells):,}")
print(f"Spells crossing into claiming    : {int(ip_spells['crossed_to_claiming'].sum()):,}")

Total sickness spells simulated : 6,470
Spells crossing into claiming    : 2,273


In [4]:
def calculate_base_incidence_rates(population_df, spells_df):
    """TOOL 1 — Base incidence table: annual probability of Healthy -> Sick(deferred), rated by
    age band AND occupation class (two standard rating factors), pooled across prior-episode
    history. Crude central-exposure rate only; no graduation or smoothing.

    Exposure is Healthy + Sick(deferred) time — i.e. the premium-paying period under
    waiver-of-premium (premium is charged while healthy or in deferred, waived once claiming
    starts). This is what makes the resulting rate an exact breakeven rate against total claims
    cost — see calculate_premium in notebook 04/05 for the algebra."""
    population_df = population_df.copy()
    population_df['age_band'] = population_df['age'].apply(age_band)

    incidence_table = {}
    exposure_table = {}
    for a_band in [0, 1, 2]:
        for occ in OCCUPATION_CLASSES:
            lives_in_cell = population_df[
                (population_df['age_band'] == a_band) & (population_df['occupation'] == occ)]
            n_lives_cell = len(lives_in_cell)
            total_possible_weeks_cell = n_lives_cell * STUDY_WEEKS
            spells_in_cell = spells_df[
                (spells_df['age_band'] == a_band) & (spells_df['occupation'] == occ)]
            no_premium_weeks_cell = spells_in_cell['claiming_weeks'].sum()
            healthy_exposure_years_cell = (total_possible_weeks_cell - no_premium_weeks_cell) / 52
            incidence_cell = (len(spells_in_cell) / healthy_exposure_years_cell
                               if healthy_exposure_years_cell > 0 else float('nan'))
            incidence_table[(a_band, occ)] = round(incidence_cell, 4)
            exposure_table[(a_band, occ)] = round(healthy_exposure_years_cell, 1)

    return {
        'method': 'central_exposure_crude_no_graduation_age_occupation_incidence',
        'states': ['healthy', 'sick_deferred', 'sick_claiming', 'death'],
        'age_band_labels': AGE_BAND_LABELS,
        'occupation_classes': OCCUPATION_CLASSES,
        'incidence_table': incidence_table,
        'incidence_exposure_life_years': exposure_table,
    }


def calculate_deferred_period_table(spells_df, options=STANDARD_DEFERRED_OPTIONS):
    """TOOL 2 — Restates the ONE reference dataset (generated once, at the real 13-week deferred
    period) under each deferred-period option, rather than re-simulating. Every spell's total
    sickness duration (weeks_in_deferred + claiming_weeks) is recorded exactly, regardless of
    whether it happened to cross into claiming — so any deferred-period cutoff can be applied to
    that one duration directly. FREQUENCY = P(cross), SEVERITY = average duration and cost of
    the claims that do."""
    total_duration = spells_df['weeks_in_deferred'] + spells_df['claiming_weeks']
    rows = []
    for D in options:
        crosses = total_duration > D
        claiming_weeks_D = total_duration[crosses] - D
        cost_D = (claiming_weeks_D / 4.345) * spells_df.loc[crosses, 'monthly_claim_amount']
        rows.append({
            'deferred_weeks': D,
            'p_cross_to_claiming': round(float(crosses.mean()), 3),
            'avg_claiming_weeks': round(float(claiming_weeks_D.mean()), 1),
            'avg_claim_cost': round(float(cost_D.mean()), 0),
        })
    return pd.DataFrame(rows).set_index('deferred_weeks')


def calculate_experience_loading(spells_df, base_table, credibility_k=40):
    """TOOL 3 — Experience-rating loading by prior-episode band, Buhlmann-style partial
    credibility blended toward 1.0. This table is deliberately separate from Tool 1 — it is
    applied AFTER the pooled base premium.

    IMPORTANT: observed cost is averaged over ALL spells in the band, not just the ones that
    crossed into claiming. A spell that recovered during the deferred period contributes a
    claim cost of 0 — this is what makes the loading capture BOTH how likely a spell is to
    reach claiming at all, and how long it runs once it does."""
    pooled_avg_claim_per_spell = spells_df['total_claim_paid'].mean()

    rows = []
    for band in [0, 1, 2]:
        band_spells = spells_df[spells_df['episode_band'] == band]
        n = len(band_spells)
        observed_avg_claim = float(band_spells['total_claim_paid'].mean()) if n else 0.0
        observed_ratio = observed_avg_claim / pooled_avg_claim_per_spell if pooled_avg_claim_per_spell else 1.0
        Z = n / (n + credibility_k)
        loading = Z * observed_ratio + (1 - Z) * 1.0
        band_label = str(band) if band < 2 else '2+'
        rows.append({
            'prior_episodes_band': band_label,
            'n_spells': n,
            'observed_avg_claim_cost': round(observed_avg_claim, 0),
            'observed_ratio': round(observed_ratio, 3),
            'credibility_Z': round(Z, 3),
            'loading_factor': round(loading, 3),
        })
    return pd.DataFrame(rows).sort_values('prior_episodes_band').reset_index(drop=True)


ip_base_table = calculate_base_incidence_rates(ip_population, ip_spells)
ip_deferred_period_table = calculate_deferred_period_table(ip_spells)
ip_loading_table = calculate_experience_loading(ip_spells, ip_base_table)

print("Tool 1 (incidence, age x occupation):")
for k, v in ip_base_table['incidence_table'].items():
    print(f"  {AGE_BAND_LABELS[k[0]]} / {k[1]}: {v:.2%}")
print("\nTool 2 (deferred-period table):")
print(ip_deferred_period_table)
print("\nTool 3 (experience loading):")
print(ip_loading_table)

Tool 1 (incidence, age x occupation):
  25-34 / desk: 4.75%
  25-34 / manual: 7.43%
  35-49 / desk: 7.95%
  35-49 / manual: 13.66%
  50-60 / desk: 13.08%
  50-60 / manual: 25.36%

Tool 2 (deferred-period table):
                p_cross_to_claiming  avg_claiming_weeks  avg_claim_cost
deferred_weeks                                                         
4                             0.593                23.0        278090.0
13                            0.340                28.2        339353.0
26                            0.198                31.5        376406.0
52                            0.082                34.6        410208.0

Tool 3 (experience loading):
  prior_episodes_band  n_spells  observed_avg_claim_cost  observed_ratio  credibility_Z  loading_factor
0                   0      2983                  96766.0           0.839          0.987           0.841
1                   1      1648                 126331.0           1.095          0.976           1.093
2             

## Part B — PMI Artifacts

A simpler product than IP: no multi-state model, no deferred period. Just **frequency × severity = pure premium**, rated by age band and sum-insured band, with an NCB (no-claim bonus) discount ladder. **No expense or profit loading** — this is deliberately the pure risk premium only, a simplification distinct from the fuller PMI Pricing Logic Explainer used earlier in the seminar.

One thing worth being explicit about: age-band frequency and sum-insured-band severity are **estimated from the synthetic dataset below**, the same "estimate from experience" spirit as the IP notebook. The NCB discount ladder is **not** — a no-claim bonus is a business/product-design rule (a discount for staying claim-free), not something you'd fit from a single cross-section of this year's claims, so it's set directly as a governance table instead.

In [5]:
PMI_N_POLICIES = 20000

PMI_AGE_BAND_LABELS = {0: '18-35', 1: '36-50', 2: '51-65'}
PMI_AGE_BAND_EDGES = [18, 36, 51, 66]  # right-open bins matching the labels above

# Illustrative age-band frequency — steadily increasing with age, same rating-factor spirit
# as the IP notebook's AGE_BASE_INCIDENCE. Calibrated (together with the age mix below) so the
# portfolio-weighted average lands close to the canonical 6% figure used across this seminar.
PMI_FREQ_BY_AGE_BAND = {0: 0.034, 1: 0.064, 2: 0.093}

PMI_SI_BAND_LABELS = {0: '3L', 1: '5L', 2: '10L', 3: '20L', 4: '50L'}
PMI_SI_BAND_VALUES = {0: 300000, 1: 500000, 2: 1000000, 3: 2000000, 4: 5000000}
# Severity multiplier relative to the 10L band, which is anchored at the canonical Rs 85,000
# figure. Population is weighted toward 10L (see PMI_SI_MIX below) so the portfolio-weighted
# average severity also lands close to Rs 85,000.
PMI_SEVERITY_BASE = 85000
PMI_SI_SEVERITY_MULT = {0: 0.75, 1: 0.85, 2: 1.00, 3: 1.25, 4: 1.60}
PMI_SI_MIX = {0: 0.15, 1: 0.20, 2: 0.35, 3: 0.20, 4: 0.10}  # concentrated at 10L, the reference band

# NCB (no-claim bonus) discount ladder — a governance/product rule, not fit from data (see note
# above). Cumulative discount for consecutive claim-free years.
PMI_NCB_TIERS = [0, 10, 20, 30, 40, 50]
PMI_NCB_DISCOUNT = {0: 1.00, 10: 0.95, 20: 0.90, 30: 0.85, 40: 0.80, 50: 0.75}
PMI_NCB_MIX = {0: 0.30, 10: 0.20, 20: 0.18, 30: 0.15, 40: 0.10, 50: 0.07}  # skewed toward newer policies


def pmi_age_band(age):
    for i, (lo, hi) in enumerate(zip(PMI_AGE_BAND_EDGES[:-1], PMI_AGE_BAND_EDGES[1:])):
        if lo <= age < hi:
            return i
    return len(PMI_AGE_BAND_EDGES) - 2  # clamp anything at/above the top edge


def generate_pmi_population(n=PMI_N_POLICIES, seed=42):
    rng = np.random.default_rng(seed)
    ages = rng.integers(18, 66, size=n)
    si_bands = rng.choice(list(PMI_SI_BAND_LABELS), size=n, p=[PMI_SI_MIX[b] for b in PMI_SI_BAND_LABELS])
    ncb_tiers = rng.choice(PMI_NCB_TIERS, size=n, p=[PMI_NCB_MIX[t] for t in PMI_NCB_TIERS])
    return pd.DataFrame({
        'policy_id': [f'PMI{str(i).zfill(5)}' for i in range(n)],
        'age': ages,
        'age_band': [pmi_age_band(a) for a in ages],
        'sum_insured_band': si_bands,
        'ncb_tier': ncb_tiers,
    })


def simulate_pmi_claims(population, seed=11):
    """One policy year per row: did a claim occur (frequency), and if so, how much (severity).
    Severity is drawn from a lognormal calibrated so its mean matches the SI-band multiplier
    table above — this mirrors real PMI experience, where claim size scales loosely with cover."""
    rng = np.random.default_rng(seed)
    records = []
    for _, pol in population.iterrows():
        freq = PMI_FREQ_BY_AGE_BAND[pol['age_band']]
        claim_occurred = rng.random() < freq
        if claim_occurred:
            mean_sev = PMI_SEVERITY_BASE * PMI_SI_SEVERITY_MULT[pol['sum_insured_band']]
            # lognormal parameterised so the mean matches mean_sev, sigma=0.5 for a realistic spread
            sigma = 0.5
            mu = np.log(mean_sev) - 0.5 * sigma ** 2
            claim_amount = float(rng.lognormal(mean=mu, sigma=sigma))
        else:
            claim_amount = 0.0
        records.append({
            'policy_id': pol['policy_id'],
            'age_band': pol['age_band'],
            'sum_insured_band': pol['sum_insured_band'],
            'ncb_tier': pol['ncb_tier'],
            'claim_occurred': claim_occurred,
            'claim_amount': round(claim_amount, 0),
        })
    return pd.DataFrame(records)


pmi_population = generate_pmi_population()
pmi_claims = simulate_pmi_claims(pmi_population)
print(f"PMI policies: {len(pmi_population):,}")
print(f"Portfolio frequency (observed): {pmi_claims['claim_occurred'].mean():.2%}")
print(f"Portfolio severity (observed):  Rs {pmi_claims.loc[pmi_claims['claim_occurred'], 'claim_amount'].mean():,.0f}")
print(f"Portfolio pure premium:         Rs {pmi_claims['claim_occurred'].mean() * pmi_claims.loc[pmi_claims['claim_occurred'], 'claim_amount'].mean():,.0f}")

PMI policies: 20,000
Portfolio frequency (observed): 6.11%
Portfolio severity (observed):  Rs 92,241
Portfolio pure premium:         Rs 5,631


In [6]:
def calculate_pmi_frequency_table(claims_df):
    """PMI TOOL 1 — annual claim frequency by age band, estimated directly from experience."""
    rows = []
    for band in sorted(PMI_AGE_BAND_LABELS):
        band_claims = claims_df[claims_df['age_band'] == band]
        rows.append({
            'age_band': PMI_AGE_BAND_LABELS[band],
            'n_policies': len(band_claims),
            'frequency': round(float(band_claims['claim_occurred'].mean()), 4),
        })
    return pd.DataFrame(rows)


def calculate_pmi_severity_table(claims_df):
    """PMI TOOL 2 — average claim severity by sum-insured band, estimated directly from
    experience (among policies that actually claimed)."""
    rows = []
    for band in sorted(PMI_SI_BAND_LABELS):
        band_claims = claims_df[(claims_df['sum_insured_band'] == band) & claims_df['claim_occurred']]
        rows.append({
            'sum_insured_band': PMI_SI_BAND_LABELS[band],
            'n_claims': len(band_claims),
            'avg_severity': round(float(band_claims['claim_amount'].mean()), 0) if len(band_claims) else float('nan'),
        })
    return pd.DataFrame(rows)


def calculate_pmi_ncb_table():
    """PMI TOOL 3 — NCB discount ladder. A governance rule, not estimated from this dataset
    (see Part B's opening note)."""
    return pd.DataFrame([
        {'ncb_tier': t, 'discount_factor': PMI_NCB_DISCOUNT[t]} for t in PMI_NCB_TIERS
    ])


pmi_frequency_table = calculate_pmi_frequency_table(pmi_claims)
pmi_severity_table = calculate_pmi_severity_table(pmi_claims)
pmi_ncb_table = calculate_pmi_ncb_table()

print("PMI Tool 1 (frequency by age band):")
print(pmi_frequency_table)
print("\nPMI Tool 2 (severity by sum-insured band):")
print(pmi_severity_table)
print("\nPMI Tool 3 (NCB discount ladder):")
print(pmi_ncb_table)

PMI Tool 1 (frequency by age band):
  age_band  n_policies  frequency
0    18-35        7584     0.0340
1    36-50        6271     0.0622
2    51-65        6145     0.0932

PMI Tool 2 (severity by sum-insured band):
  sum_insured_band  n_claims  avg_severity
0               3L       167       67279.0
1               5L       234       77028.0
2              10L       431       87778.0
3              20L       261      108618.0
4              50L       128      134252.0

PMI Tool 3 (NCB discount ladder):
   ncb_tier  discount_factor
0         0             1.00
1        10             0.95
2        20             0.90
3        30             0.85
4        40             0.80
5        50             0.75


## Save Artifacts

Both files below are everything notebook 05 needs — it will load these and never touch `generate_population`, `simulate_spells`, `generate_pmi_population`, or `simulate_pmi_claims` directly.

In [7]:
ip_artifacts = {
    'population': ip_population,
    'spells': ip_spells,
    'base_table': ip_base_table,
    'deferred_period_table': ip_deferred_period_table,
    'loading_table': ip_loading_table,
}
with open(OUTPUT_DIR / 'ip_pricing_artifacts.pkl', 'wb') as f:
    pickle.dump(ip_artifacts, f)

pmi_artifacts = {
    'population': pmi_population,
    'claims': pmi_claims,
    'frequency_table': pmi_frequency_table,
    'severity_table': pmi_severity_table,
    'ncb_table': pmi_ncb_table,
}
with open(OUTPUT_DIR / 'pmi_pricing_artifacts.pkl', 'wb') as f:
    pickle.dump(pmi_artifacts, f)

print(f"Saved ip_pricing_artifacts.pkl  ({(OUTPUT_DIR / 'ip_pricing_artifacts.pkl').stat().st_size:,} bytes)")
print(f"Saved pmi_pricing_artifacts.pkl ({(OUTPUT_DIR / 'pmi_pricing_artifacts.pkl').stat().st_size:,} bytes)")
print("\nBoth files are ready for notebook 05. If you're running notebooks in separate Colab")
print("sessions, upload these two files to the shared repo's data/ folder before starting 05.")

Saved ip_pricing_artifacts.pkl  (1,275,691 bytes)
Saved pmi_pricing_artifacts.pkl (1,943,522 bytes)

Both files are ready for notebook 05. If you're running notebooks in separate Colab
sessions, upload these two files to the shared repo's data/ folder before starting 05.
